In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Fetch dataset from UCI
credit = fetch_ucirepo(id=350)

X = credit.data.features
y = credit.data.targets

# Combine into one dataframe
df = pd.concat([X, y], axis=1)

print(df.shape)
print(df.columns.tolist())

(30000, 24)
['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'Y']


In [2]:
df.columns = [
    'credit_limit', 'sex', 'education', 'marriage', 'age',
    'pay_sept', 'pay_aug', 'pay_jul', 'pay_jun', 'pay_may', 'pay_apr',
    'bill_sept', 'bill_aug', 'bill_jul', 'bill_jun', 'bill_may', 'bill_apr',
    'paid_sept', 'paid_aug', 'paid_jul', 'paid_jun', 'paid_may', 'paid_apr',
    'default'
]

print(df.columns.tolist())
df.head(3)

['credit_limit', 'sex', 'education', 'marriage', 'age', 'pay_sept', 'pay_aug', 'pay_jul', 'pay_jun', 'pay_may', 'pay_apr', 'bill_sept', 'bill_aug', 'bill_jul', 'bill_jun', 'bill_may', 'bill_apr', 'paid_sept', 'paid_aug', 'paid_jul', 'paid_jun', 'paid_may', 'paid_apr', 'default']


,credit_limit,sex,education,marriage,age,pay_sept,pay_aug,pay_jul,pay_jun,pay_may,...,bill_jun,bill_may,bill_apr,paid_sept,paid_aug,paid_jul,paid_jun,paid_may,paid_apr,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


In [3]:
import sqlite3

# Create database file and write the table
conn = sqlite3.connect('../data/credit.db')
df.to_sql('clients', conn, if_exists='replace', index=False)

# Verify it worked
result = pd.read_sql('SELECT COUNT(*) AS row_count FROM clients', conn)
print(result)

   row_count
0      30000


In [4]:
query = """
SELECT
    default AS defaulted,
    COUNT(*) AS num_clients,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clients), 2) AS pct
FROM clients
GROUP BY default
"""

pd.read_sql(query, conn)

DatabaseError: Execution failed on sql '
SELECT 
    default AS defaulted,
    COUNT(*) AS num_clients,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clients), 2) AS pct
FROM clients
GROUP BY default
': near "default": syntax error

In [5]:
query = """
SELECT
    "default" AS defaulted,
    COUNT(*) AS num_clients,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clients), 2) AS pct
FROM clients
GROUP BY "default"
"""

pd.read_sql(query, conn)

,defaulted,num_clients,pct
0,0,23364,77.88
1,1,6636,22.12


In [6]:
df = df.rename(columns={'default': 'defaulted'})
df.to_sql('clients', conn, if_exists='replace', index=False)

query = """
SELECT
    defaulted,
    COUNT(*) AS num_clients,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clients), 2) AS pct
FROM clients
GROUP BY defaulted
"""

pd.read_sql(query, conn)

,defaulted,num_clients,pct
0,0,23364,77.88
1,1,6636,22.12


In [7]:
query = """
SELECT
    pay_sept AS repayment_status,
    COUNT(*) AS num_clients,
    SUM(defaulted) AS num_defaults,
    ROUND(AVG(defaulted) * 100, 2) AS default_rate_pct
FROM clients
GROUP BY pay_sept
ORDER BY pay_sept
"""

pd.read_sql(query, conn)

,repayment_status,num_clients,num_defaults,default_rate_pct
0,-2,2759,365,13.23
1,-1,5686,954,16.78
2,0,14737,1888,12.81
3,1,3688,1252,33.95
4,2,2667,1844,69.14
5,3,322,244,75.78
6,4,76,52,68.42
7,5,26,13,50.00
8,6,11,6,54.55
9,7,9,7,77.78


In [8]:
query = """
SELECT
    CASE
        WHEN age < 30 THEN '20s'
        WHEN age < 40 THEN '30s'
        WHEN age < 50 THEN '40s'
        WHEN age < 60 THEN '50s'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS num_clients,
    ROUND(AVG(defaulted) * 100, 2) AS default_rate_pct,
    ROUND(AVG(credit_limit), 0) AS avg_credit_limit
FROM clients
GROUP BY age_group
ORDER BY age_group
"""

pd.read_sql(query, conn)

,age_group,num_clients,default_rate_pct,avg_credit_limit
0,20s,9618,22.84,124209.0
1,30s,11238,20.25,197001.0
2,40s,6464,22.97,180786.0
3,50s,2341,24.86,163909.0
4,60+,339,28.32,187847.0


In [9]:
queries = """
-- Query 1: Overall default rate
SELECT
    defaulted,
    COUNT(*) AS num_clients,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM clients), 2) AS pct
FROM clients
GROUP BY defaulted;

-- Query 2: Default rate by September repayment status
SELECT
    pay_sept AS repayment_status,
    COUNT(*) AS num_clients,
    SUM(defaulted) AS num_defaults,
    ROUND(AVG(defaulted) * 100, 2) AS default_rate_pct
FROM clients
GROUP BY pay_sept
ORDER BY pay_sept;

-- Query 3: Default rate and average credit limit by age group
SELECT
    CASE
        WHEN age < 30 THEN '20s'
        WHEN age < 40 THEN '30s'
        WHEN age < 50 THEN '40s'
        WHEN age < 60 THEN '50s'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS num_clients,
    ROUND(AVG(defaulted) * 100, 2) AS default_rate_pct,
    ROUND(AVG(credit_limit), 0) AS avg_credit_limit
FROM clients
GROUP BY age_group
ORDER BY age_group;
"""

with open('../queries.sql', 'w') as f:
    f.write(queries)

print("Saved!")

Saved!
